# Week 4 Task 3 — Reproducible Fare Prediction Pipeline

## Goal

Build a complete Spark ML workflow for trip-level fare prediction by directly reusing the feature dataset and fitted feature pipeline saved by Task 2.

This notebook:

- loads and validates Task 2's chronological feature Delta table;
- loads Task 2's fitted preprocessing `PipelineModel` instead of redefining its features;
- trains a regularized Spark `LinearRegression` model on Task 2's `features` vectors;
- evaluates both the model and a Train-mean baseline using MAE, RMSE, and R²;
- combines Task 2 preprocessing with the fitted regressor into one deployable `PipelineModel`; and
- writes a reproducibility manifest containing upstream paths, data coverage, parameters, metrics, and timings.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys
import time
import uuid

from delta.tables import DeltaTable
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
# Task 4 runs this same notebook on the Approach A (raw-file) dataset by setting
# WEEK4_VARIANT=raw. The default "platform" variant is Approach B.
VARIANT = os.environ.get("WEEK4_VARIANT", "platform")
assert VARIANT in {"platform", "raw"}, f"Unknown WEEK4_VARIANT: {VARIANT}"
ML_DIR = PROJECT_ROOT / "data" / "processed" / ("ml" if VARIANT == "platform" else "ml_raw")
MODEL_DIR = PROJECT_ROOT / "data" / "models" / ("" if VARIANT == "platform" else "raw")
TASK1_PATH = ML_DIR / "fare_training_dataset"
FEATURE_DATASET_PATH = ML_DIR / "fare_feature_dataset"
FEATURE_PIPELINE_PATH = MODEL_DIR / "fare_feature_pipeline"
MODEL_PATH = MODEL_DIR / "fare_prediction_pipeline"
METRICS_PATH = PROJECT_ROOT / "artifacts" / (
    "week4_fare_metrics.json" if VARIANT == "platform" else "task4_approach_a_fare_metrics.json"
)

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session
from data_intensive_computing.week4.modeling import (
    build_regressor,
    evaluate_split,
    hyperparameters,
)

# Keep both heap and native-memory pressure below the Windows page-file limit.
# Evaluation is deliberately streaming and does not cache transformed rows.
spark = build_spark_session(
    "Week4-Task3-Reproducible-Fare-Pipeline",
    driver_memory="4g",
    shuffle_partitions=8,
    master="local[2]",
)
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)

assert spark.sparkContext.master == "local[2]", (
    "An older SparkContext is still active. Restart the notebook kernel, then "
    "run Task 3 from the first cell so the local[2] resource limit is applied."
)

print("Variant:", VARIANT)
print("Task 1 input:", TASK1_PATH)
print("Task 2 feature dataset:", FEATURE_DATASET_PATH)
print("Task 2 feature pipeline:", FEATURE_PIPELINE_PATH)
print("Complete model output:", MODEL_PATH)
print("Metrics output:", METRICS_PATH)
print("Spark master:", spark.sparkContext.master)

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Task 1 input: D:\Master\data intensive\data-intensive-computing\data\processed\ml\fare_training_dataset
Task 2 feature dataset: D:\Master\data intensive\data-intensive-computing\data\processed\ml\fare_feature_dataset
Task 2 feature pipeline: D:\Master\data intensive\data-intensive-computing\data\models\fare_feature_pipeline
Complete model output: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Metrics output: D:\Master\data intensive\data-intensive-computing\artifacts\week4_fare_metrics.json
Spark master: local[2]


## 1. Load and validate the Task 2 feature dataset

The model predicts the non-negative base fare in US dollars. A regularized linear model is appropriate because the objective is a transparent and reproducible workflow rather than maximum predictive accuracy.

Task 2 has already fitted missing-value handling, categorical encoding, robust scaling, and vector assembly on Train only. Task 3 loads its partitioned Delta output containing `label` and `features`; no random split or feature redefinition is performed here.

In [2]:
EXPECTED_SPLITS = {"train", "validation", "test"}

feature_data = spark.read.format("delta").load(str(FEATURE_DATASET_PATH))
required_columns = {"trip_id", "pickup_datetime", "split", "label", "features"}
missing_columns = sorted(required_columns - set(feature_data.columns))
assert not missing_columns, f"Task 2 feature dataset is missing columns: {missing_columns}"

split_profile_rows = (
    feature_data.groupBy("split")
    .agg(
        F.count("*").alias("row_count"),
        F.min("pickup_datetime").alias("first_pickup"),
        F.max("pickup_datetime").alias("last_pickup"),
        F.avg("label").alias("mean_label"),
        F.sum(
            (
                F.col("label").isNull()
                | F.isnan("label")
                | (F.col("label") < 0)
            ).cast("long")
        ).alias("invalid_labels"),
    )
    .orderBy("split")
    .collect()
)

assert {row["split"] for row in split_profile_rows} == EXPECTED_SPLITS
assert all(row["row_count"] > 0 for row in split_profile_rows)
assert all(row["invalid_labels"] == 0 for row in split_profile_rows)

split_profile = {
    row["split"]: {
        "row_count": row["row_count"],
        "first_pickup": row["first_pickup"].isoformat(),
        "last_pickup": row["last_pickup"].isoformat(),
        "mean_label": row["mean_label"],
    }
    for row in split_profile_rows
}

# Detect a stale Task 2 output after Task 1 has been rebuilt.
task1_split_counts = {
    row["split"]: row["count"]
    for row in (
        spark.read.format("delta")
        .load(str(TASK1_PATH))
        .groupBy("split")
        .count()
        .collect()
    )
}
feature_split_counts = {
    split_name: profile["row_count"]
    for split_name, profile in split_profile.items()
}
assert feature_split_counts == task1_split_counts, (
    "Task 2 feature data is stale or incomplete. Rerun w4_task2.ipynb after "
    f"Task 1. Task 1 counts={task1_split_counts}, "
    f"Task 2 counts={feature_split_counts}."
)

feature_dimension = (
    feature_data.select(F.size(vector_to_array("features")).alias("size"))
    .first()["size"]
)

# The table is partitioned by split, so these filters are partition-pruned.
train_features = feature_data.filter(F.col("split") == "train")
validation_features = feature_data.filter(F.col("split") == "validation")
test_features = feature_data.filter(F.col("split") == "test")

print(json.dumps(split_profile, indent=2))
print("Task 2 feature dimension:", feature_dimension)

{
  "test": {
    "row_count": 1783894,
    "first_pickup": "2024-03-16T05:00:00",
    "last_pickup": "2024-04-01T05:59:59",
    "mean_label": 19.412305108935886
  },
  "train": {
    "row_count": 5870362,
    "first_pickup": "2024-01-01T06:00:00",
    "last_pickup": "2024-03-01T05:59:58",
    "mean_label": 18.54631074880932
  },
  "validation": {
    "row_count": 1726201,
    "first_pickup": "2024-03-01T06:00:00",
    "last_pickup": "2024-03-16T04:59:59",
    "mean_label": 19.015878956158584
  }
}
Task 2 feature dimension: 561


## 2. Load the fitted Task 2 feature pipeline

Task 2 is the single source of truth for feature selection and preprocessing. Its saved `PipelineModel` contains the Train-fitted medians, category mappings, one-hot dimensions, robust-scaling statistics, and final vector ordering. Task 3 loads these fitted stages rather than copying their definitions.

In [3]:
feature_pipeline_model = PipelineModel.load(str(FEATURE_PIPELINE_PATH))

print("Loaded Task 2 feature stages:")
for position, stage in enumerate(feature_pipeline_model.stages, start=1):
    print(f"{position}. {stage.__class__.__name__}")

Loaded Task 2 feature stages:
1. SQLTransformer
2. ImputerModel
3. StringIndexerModel
4. OneHotEncoderModel
5. VectorAssembler
6. RobustScalerModel
7. VectorAssembler


## 3. Train the regression model and assemble the complete pipeline

Linear regression is fitted directly on Task 2's persisted Train feature vectors. After fitting, the regression model is appended to the loaded Task 2 transformer stages and saved as one complete `PipelineModel`. The complete model therefore accepts Task 1-style raw rows for future prediction while Task 3 avoids refitting or redefining preprocessing.

Retraining now has an explicit dependency order: rebuild Task 1, rerun Task 2 so its feature pipeline is refitted on the new Train split, and then rerun Task 3.

In [4]:
# The regressor configuration is shared with Task 4 through week4/modeling.py.
regressor = build_regressor()

run_id = str(uuid.uuid4())
run_started = time.perf_counter()
fit_started = time.perf_counter()
regression_model = regressor.fit(train_features)
fit_seconds = time.perf_counter() - fit_started

# Combine Task 2's fitted transformers with the fitted regressor. The resulting
# model can transform raw Task 1 rows and then predict in one operation.
complete_pipeline_model = PipelineModel(
    stages=[*feature_pipeline_model.stages, regression_model]
)

save_started = time.perf_counter()
complete_pipeline_model.write().overwrite().save(str(MODEL_PATH))
save_seconds = time.perf_counter() - save_started

print("Run ID:", run_id)
print("Fit seconds:", round(fit_seconds, 4))
print("Save seconds:", round(save_seconds, 4))
print("Model saved to:", MODEL_PATH)
print("Complete stages:")
for position, stage in enumerate(complete_pipeline_model.stages, start=1):
    print(f"{position}. {stage.__class__.__name__}")

Run ID: 748eb2e2-ba1d-46ad-a29a-066823178b95
Fit seconds: 88.2416
Save seconds: 7.7032
Model saved to: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Complete stages:
1. SQLTransformer
2. ImputerModel
3. StringIndexerModel
4. OneHotEncoderModel
5. VectorAssembler
6. RobustScalerModel
7. VectorAssembler
8. LinearRegressionModel


## 4. Evaluate Validation and Test

MAE is directly interpretable in dollars, RMSE gives additional weight to large fare errors, and R² measures explained variation. The Train-mean baseline shows whether the pipeline improves on a trivial reproducible predictor. Test is used only for final reporting, not parameter selection.

In [5]:
# evaluate_split() is imported from week4/modeling.py.
evaluation_started = time.perf_counter()
train_mean_label = float(split_profile["train"]["mean_label"])
validation_metrics, baseline_validation_metrics = evaluate_split(
    regression_model, validation_features, train_mean_label
)
test_metrics, baseline_test_metrics = evaluate_split(
    regression_model, test_features, train_mean_label
)
evaluation_seconds = time.perf_counter() - evaluation_started

metrics = {
    "model": {
        "validation": validation_metrics,
        "test": test_metrics,
    },
    "train_mean_baseline": {
        "prediction": train_mean_label,
        "validation": baseline_validation_metrics,
        "test": baseline_test_metrics,
    },
}

print(json.dumps(metrics, indent=2))
(
    regression_model.transform(test_features.limit(10))
    .select("trip_id", "pickup_datetime", "label", "prediction")
    .show(10, truncate=False)
)

{
  "model": {
    "validation": {
      "mae": 6.8062108692577645,
      "rmse": 10.548761118778405,
      "r2": 0.6118116366043727
    },
    "test": {
      "mae": 6.996983272326323,
      "rmse": 10.810090162072443,
      "r2": 0.617763966297236
    }
  },
  "train_mean_baseline": {
    "prediction": 18.54631074880932,
    "validation": {
      "mae": 10.89899306584715,
      "rmse": 16.93741645174195,
      "r2": -0.0007691959106215229
    },
    "test": {
      "mae": 11.190630015547175,
      "rmse": 17.506346495024125,
      "r2": -0.00245303147480036
    }
  }
}
+----------------------------------------------------------------+-------------------+-----+------------------+
|trip_id                                                         |pickup_datetime    |label|prediction        |
+----------------------------------------------------------------+-------------------+-----+------------------+
|00017842c00dd912671c049d5c0f24785678b3af2cc288223ac917c19edd5a8f|2024-03-28 15:41:53|

In [6]:
for split in ("validation", "test"):
    model_result = metrics["model"][split]
    baseline_result = metrics["train_mean_baseline"][split]
    mae_reduction = 100.0 * (
        baseline_result["mae"] - model_result["mae"]
    ) / baseline_result["mae"]
    rmse_reduction = 100.0 * (
        baseline_result["rmse"] - model_result["rmse"]
    ) / baseline_result["rmse"]
    print(
        f"{split}: MAE reduced by {mae_reduction:.2f}%, "
        f"RMSE reduced by {rmse_reduction:.2f}%, "
        f"R²={model_result['r2']:.3f}"
    )

mae_gap_pct = 100.0 * (
    metrics["model"]["test"]["mae"]
    - metrics["model"]["validation"]["mae"]
) / metrics["model"]["validation"]["mae"]
print(f"Validation-to-test MAE increase: {mae_gap_pct:.2f}%")

validation: MAE reduced by 37.55%, RMSE reduced by 37.72%, R²=0.612
test: MAE reduced by 37.47%, RMSE reduced by 38.25%, R²=0.618
Validation-to-test MAE increase: 2.80%


## 5. Write the reproducibility manifest

The complete model was saved immediately after fitting so an evaluation failure cannot discard the training result. The model directory is overwritten intentionally during retraining. The manifest records both upstream Delta versions, the Task 2 feature pipeline path and vector dimension, split coverage, model parameters, Spark configuration, metrics, and phase timings.

In [7]:
task1_version = int(
    DeltaTable.forPath(spark, str(TASK1_PATH))
    .history(1)
    .select("version")
    .first()["version"]
)
feature_dataset_version = int(
    DeltaTable.forPath(spark, str(FEATURE_DATASET_PATH))
    .history(1)
    .select("version")
    .first()["version"]
)
spark_conf = spark.sparkContext.getConf()

run_manifest = {
    "run_id": run_id,
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "prediction_task": "fare_amount_regression",
    "variant": VARIANT,
    "target": "fare_amount_usd",
    "algorithm": "Spark LinearRegression",
    "input": {
        "task1_training_dataset": {
            "path": str(TASK1_PATH),
            "delta_version": task1_version,
        },
        "task2_feature_dataset": {
            "path": str(FEATURE_DATASET_PATH),
            "delta_version": feature_dataset_version,
            "splits": split_profile,
        },
    },
    "features": {
        "source": "Task 2 fitted feature pipeline",
        "pipeline_path": str(FEATURE_PIPELINE_PATH),
        "vector_dimension": feature_dimension,
    },
    "hyperparameters": hyperparameters(),
    "metrics": metrics,
    "timings_seconds": {
        "fit": round(fit_seconds, 4),
        "evaluate": round(evaluation_seconds, 4),
        "save": round(save_seconds, 4),
        "end_to_end": round(time.perf_counter() - run_started, 4),
    },
    "model_path": str(MODEL_PATH),
    "pipeline_stages": [
        stage.__class__.__name__ for stage in complete_pipeline_model.stages
    ],
    "spark": {
        "version": spark.version,
        "master": spark_conf.get("spark.master", ""),
        "driver_memory": spark_conf.get("spark.driver.memory", ""),
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "session_time_zone": spark.conf.get("spark.sql.session.timeZone"),
    },
}

METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True, default=str) + "\n",
    encoding="utf-8",
)

print("Complete PipelineModel saved to:", MODEL_PATH)
print("Run manifest saved to:", METRICS_PATH)

Complete PipelineModel saved to: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Run manifest saved to: D:\Master\data intensive\data-intensive-computing\artifacts\week4_fare_metrics.json


## 6. Verify loading and document retraining

Loading the saved object proves that Task 2 preprocessing and Task 3 regression were persisted together. Verification uses raw Task 1 rows because the complete model must perform preprocessing before prediction. For retraining, refresh the integrated platform and rerun Task 1, Task 2, and Task 3 in order.

In [8]:
reloaded_model = PipelineModel.load(str(MODEL_PATH))
task1_test_sample = (
    spark.read.format("delta")
    .load(str(TASK1_PATH))
    .filter(F.col("split") == "test")
    .limit(100)
)
reloaded_sample = reloaded_model.transform(task1_test_sample).select(
    "trip_id", "label", "prediction"
)
assert reloaded_sample.count() == 100
assert reloaded_sample.filter(
    F.col("prediction").isNull() | F.isnan("prediction")
).count() == 0

print("Reloaded model stages:", len(reloaded_model.stages))
print("Complete model transformed 100 raw Task 1 rows successfully.")

Reloaded model stages: 8
Complete model transformed 100 raw Task 1 rows successfully.


## 7. Reusability and extension discussion

### Which concrete modules are reusable?

1. **Upstream artifact contract** — Task 3 requires Task 2's Delta table with `trip_id`, `pickup_datetime`, `split`, `label`, and `features`, plus its saved feature `PipelineModel`.
2. **Data-loading and validation module** — required-column, split, label, date-range, and feature-dimension checks fail early when an upstream artifact is incompatible.
3. **Training module** — the fixed `LinearRegression` configuration can be fitted on any compatible Task 2 feature dataset.
4. **Evaluation and baseline module** — `evaluate_split()` computes model and Train-mean baseline MAE, RMSE, and R² in one Spark aggregation.
5. **Timing module** — fitting, evaluation, saving, and end-to-end phases use stable `time.perf_counter()` boundaries.
6. **Persistence module** — Task 2 transformer stages and the Task 3 regression model are combined into one complete `PipelineModel`, then saved, loaded, and tested on raw Task 1 rows.
7. **Experiment-manifest module** — the JSON artifact records both upstream versions, the feature pipeline path, feature dimension, parameters, metrics, timings, stages, and Spark configuration.

### How can new features be incorporated?

Add the source field to Task 1 and update the feature definitions only in Task 2. Rerun Task 2 to refit preprocessing and regenerate its feature table, then rerun Task 3. Task 3 does not contain feature lists and therefore does not require a matching code change. Leakage-prone post-trip monetary fields must remain excluded.

### How is retraining supported?

Weeks 1–3 refresh the integrated platform. Task 1 rebuilds chronological raw training rows, Task 2 refits preprocessing and rewrites the feature dataset, and Task 3 refits only the regressor before assembling and replacing the complete saved model. The manifest records new Task 1 and Task 2 Delta versions and a new run ID.

### How could multiple prediction tasks be supported?

A new task should publish its own Task 1-style target dataset and Task 2-style fitted feature pipeline plus feature Delta table. Task 3's loading, validation, timing, persistence, and manifest pattern can remain unchanged. Regression tasks can reuse `evaluate_split()`; classification tasks replace the estimator, baseline, and evaluator. Each task should use separate upstream paths, model output, and metrics artifact.

## 8. Reusable modules for Task 4

Task 4 should implement raw and platform dataset construction separately, then reuse these Task 3 modules once each approach has produced a compatible feature table:

1. **Spark configuration module** — use the same driver memory, local threads, shuffle partitions, time zone, and file settings.
2. **Feature-table contract module** — reuse required columns, chronological split checks, invalid-label checks, split profiles, and feature-dimension checks.
3. **Training module** — fit the same `LinearRegression` configuration on both feature tables.
4. **Evaluation and baseline module** — reuse `evaluate_split()` and the same Train-mean baseline metrics.
5. **Timing module** — preserve separate boundaries for dataset construction, feature preparation, fitting, evaluation, and saving. Task 4 should add warm-up runs, repetitions, and median reporting.
6. **Persistence and verification module** — reuse model saving/loading and prediction checks.
7. **Experiment-manifest module** — reuse the same versions, split profiles, feature dimensions, parameters, metrics, timings, stages, and Spark fields, adding an `approach` value of `raw` or `platform`.

The two approaches differ before the feature-table contract: Approach A performs raw loading, cleaning, integration, and feature engineering itself, while Approach B reuses the platform and Task 2 artifacts. From model fitting onward, configuration and evaluation must remain identical.

`artifacts/week4_fare_metrics.json` is an example of the record structure that Task 4 can reuse. Its current `fit` timing is supporting evidence for the platform workflow, not a complete A/B benchmark. Task 4 should rerun both approaches with the same configuration, include one discarded warm-up, repeat each measured phase, and compare median dataset-construction and model-training times separately.